<a href="https://colab.research.google.com/github/Leashaniya/Research-Project/blob/leasha/05_RAG_QUESTION_GENERATION_(PP1%2C_MAIN_QUESTIONS_ONLY).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# # ==========================================================
# # NOTEBOOK 5 (v3) — RAG MODEL PAPER GENERATION (NO OPENAI)
# #
# # UPDATED FOR YOUR CURRENT STATE:
# #  - Works with small slide index (e.g., FAISS vectors ~163)
# #  - STRUCTURED questions ONLY (NO MCQ)
# #  - Stronger retrieval (top_k=15, context cap=2000 chars)
# #  - NO "fallback copies template" (fallback is a safe generic shell)
# #  - Cleans context and blocks slide markers and [SOURCE:]
# #  - Garbage detection + regeneration
# #
# # Output:
# #   /content/drive/MyDrive/RP/model_exam_paper.json
# #   /content/drive/MyDrive/RP/model_exam_paper.txt
# # ==========================================================

# # -------------------------------
# # 0) Mount Drive
# # -------------------------------
# from google.colab import drive
# drive.mount("/content/drive")

# # -------------------------------
# # 1) Install deps
# # -------------------------------
# !pip -q install sentence-transformers faiss-cpu transformers accelerate bitsandbytes

# import json, re, random
# from pathlib import Path
# import numpy as np
# import faiss
# from sentence_transformers import SentenceTransformer

# import torch
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# # -------------------------------
# # 2) Paths (match Notebook 2/4 outputs)
# # -------------------------------
# RP_ROOT   = Path("/content/drive/MyDrive/RP")

# TEMPLATES_PATH = RP_ROOT / "template_questions.json"
# BLUEPRINT_PATH = RP_ROOT / "exam_blueprint_template.json"

# SLIDES_CHUNKS_PATH = RP_ROOT / "lecture_slides_extraction" / "slides_chunks.jsonl"
# SLIDES_META_PATH   = RP_ROOT / "slides_embeddings" / "slides_metadata.jsonl"
# SLIDES_FAISS_PATH  = RP_ROOT / "slides_embeddings" / "slides_faiss_index_flatip.index"

# OUT_JSON = RP_ROOT / "model_exam_paper.json"
# OUT_TXT  = RP_ROOT / "model_exam_paper.txt"

# for p in [TEMPLATES_PATH, BLUEPRINT_PATH, SLIDES_CHUNKS_PATH, SLIDES_META_PATH, SLIDES_FAISS_PATH]:
#     if not p.exists():
#         raise FileNotFoundError(f"Missing required file: {p}")

# print("✅ All required files found.")

# # -------------------------------
# # 3) Load artifacts
# # -------------------------------
# templates = json.loads(TEMPLATES_PATH.read_text(encoding="utf-8"))
# exam_blueprint = json.loads(BLUEPRINT_PATH.read_text(encoding="utf-8"))

# slide_chunks = [json.loads(l) for l in SLIDES_CHUNKS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
# slide_meta   = [json.loads(l) for l in SLIDES_META_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]

# index = faiss.read_index(str(SLIDES_FAISS_PATH))
# print("✅ Loaded FAISS index. vectors:", index.ntotal)

# # -------------------------------
# # 4) Retrieval embedding model
# # -------------------------------
# # Must match Notebook 4 for best retrieval
# retriever_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# # -------------------------------
# # 5) Generator model (local)
# # -------------------------------
# # CPU-friendly default:
# FLAN_MODEL_NAME = "google/flan-t5-base"
# # If you enable GPU, you can switch to:
# # FLAN_MODEL_NAME = "google/flan-t5-large"

# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Device:", device)
# print("Loading generator:", FLAN_MODEL_NAME)

# tokenizer = AutoTokenizer.from_pretrained(FLAN_MODEL_NAME)
# model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_MODEL_NAME).to(device)
# model.eval()
# print("✅ Generator loaded.")

# # -------------------------------
# # 6) Helpers
# # -------------------------------
# def normalize_l2(x: np.ndarray) -> np.ndarray:
#     x = x.astype("float32")
#     norm = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
#     return x / norm

# BAD_CONTEXT_PATTERNS = [
#     r"\[SOURCE:.*?\]",
#     r"---\s*SLIDE\s*\d+\s*---",
#     r"---\s*PAGE\s*\d+\s*---",
# ]

# def clean_context(text: str) -> str:
#     t = text or ""
#     for pat in BAD_CONTEXT_PATTERNS:
#         t = re.sub(pat, " ", t, flags=re.I | re.S)
#     t = re.sub(r'["\']{2,}', " ", t)     # repeated quotes
#     t = re.sub(r"[-_]{3,}", " ", t)      # long dashes
#     t = re.sub(r"\s+", " ", t).strip()
#     # trim extreme length
#     if len(t) > 2500:
#         t = t[:2500].rsplit(" ", 1)[0] + "..."
#     return t.strip()

# def retrieve_slide_context(query_text: str, top_k: int = 15, max_chars: int = 2000):
#     q_emb = retriever_model.encode([query_text]).astype("float32")
#     q_emb = normalize_l2(q_emb)
#     D, I = index.search(q_emb, top_k)

#     hits = []
#     parts = []
#     used_chars = 0

#     for idx, score in zip(I[0].tolist(), D[0].tolist()):
#         if idx < 0:
#             continue
#         chunk = slide_chunks[idx] if idx < len(slide_chunks) else {}
#         meta  = slide_meta[idx] if idx < len(slide_meta) else {}

#         raw = (chunk.get("text") or "").strip()
#         cleaned = clean_context(raw)

#         # skip weak chunks
#         if not cleaned or len(cleaned.split()) < 18:
#             continue

#         hits.append({
#             "score": float(score),
#             "chunk_id": meta.get("chunk_id", chunk.get("chunk_id")),
#             "pdf_stem": meta.get("pdf_stem", chunk.get("pdf_stem")),
#             "slide_no": meta.get("slide_no", chunk.get("slide_no")),
#         })

#         parts.append(cleaned)
#         used_chars += len(cleaned) + 1
#         if used_chars >= max_chars:
#             break

#     context = "\n".join(parts).strip()
#     if len(context) > max_chars:
#         context = context[:max_chars].rsplit(" ", 1)[0] + "..."

#     return context, hits

# def looks_like_garbage(text: str) -> bool:
#     t = (text or "").strip()
#     if len(t) < 60:
#         return True
#     if "[SOURCE" in t or "--- SLIDE" in t or "--- PAGE" in t:
#         return True
#     if re.search(r'""\s*-\s*""', t):
#         return True
#     alpha = sum(c.isalpha() for c in t)
#     if alpha / max(1, len(t)) < 0.35:
#         return True
#     # too many repeated punctuation
#     if len(re.findall(r'["\']', t)) > 25:
#         return True
#     if len(re.findall(r"-", t)) > 60:
#         return True
#     return False

# def validate_structured(text: str) -> bool:
#     t = (text or "").strip()
#     has_parts = bool(re.search(r"\(a\)", t, flags=re.I) and re.search(r"\(b\)", t, flags=re.I))
#     return has_parts and (not looks_like_garbage(t))

# def enforce_clean_final(text: str) -> str:
#     t = (text or "").strip()
#     t = re.sub(r"\[SOURCE:.*?\]", " ", t, flags=re.I | re.S)
#     t = re.sub(r"---\s*SLIDE\s*\d+\s*---", " ", t, flags=re.I)
#     t = re.sub(r"---\s*PAGE\s*\d+\s*---", " ", t, flags=re.I)
#     t = re.sub(r"\s+", " ", t).strip()
#     return t

# def generate_text(prompt: str, max_new_tokens=280):
#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
#     inputs = {k: v.to(device) for k, v in inputs.items()}
#     with torch.no_grad():
#         out = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=True,
#             temperature=0.7,
#             top_p=0.9,
#             num_beams=1,
#             repetition_penalty=1.15,
#         )
#     return tokenizer.decode(out[0], skip_special_tokens=True).strip()

# def prompt_for_structured(template_text: str, context: str, pattern_label: str, marks: int):
#     return f"""
# You are a university lecturer creating ONE structured exam question for Database Management Systems worth about {marks} marks.

# STRICT RULES:
# - Use ONLY the CONTEXT facts. Do not invent additional theory.
# - Do NOT copy sentences from TEMPLATE. You must rewrite using new entity/table/attribute names and different numbers/values.
# - Output MUST include (a) and (b) subparts (and optionally (c)).
# - Output ONLY the question. No answers. No solutions.

# Pattern: {pattern_label}

# Required format:
# <Main question statement describing the scenario>
# (a) ...
# (b) ...
# (c) ... (optional)

# TEMPLATE (style reference only):
# {template_text}

# CONTEXT (must ground the question):
# {context}
# """.strip()

# # -------------------------------
# # 7) Generate ONE question per slot (STRUCTURED ONLY)
# # -------------------------------
# def generate_question(template_obj: dict, marks: int, max_tries: int = 6):
#     template_text = (template_obj.get("full_text") or "").strip()
#     pattern_label = template_obj.get("pattern_label", "GENERAL_THEORY")

#     context, hits = retrieve_slide_context(template_text, top_k=15, max_chars=2000)
#     if not context:
#         # fallback context: short trimmed template (still not copying, just grounding topic)
#         context = clean_context(template_text)[:1200]

#     for attempt in range(1, max_tries + 1):
#         prompt = prompt_for_structured(template_text, context, pattern_label, marks)
#         gen = generate_text(prompt, max_new_tokens=320)
#         gen = enforce_clean_final(gen)

#         if validate_structured(gen):
#             return gen, hits, "STRUCTURED"

#     # SAFE fallback (DOES NOT COPY TEMPLATE)
#     safe = (
#         "Consider a realistic database scenario relevant to the given topic.\n"
#         "(a) Define the relations/entities and state primary keys and any constraints.\n"
#         "(b) Write the required SQL/ER/normalization steps to solve the problem.\n"
#         "(c) Briefly justify your design/queries."
#     )
#     return safe, hits, "FALLBACK_GENERIC"

# # -------------------------------
# # 8) Assemble full paper using blueprint slots
# # -------------------------------
# slots = exam_blueprint.get("question_slots", [])
# if not slots:
#     raise ValueError("exam_blueprint_template.json has no question_slots")

# templates_by_marks = {}
# for t in templates:
#     m = t.get("marks", None)
#     if m is None:
#         continue
#     templates_by_marks.setdefault(int(m), []).append(t)

# def pick_template_for_mark(mark: int):
#     pool = templates_by_marks.get(int(mark), [])
#     if pool:
#         return random.choice(pool)
#     return random.choice(templates)

# random.seed(42)

# generated_questions = []
# print("\n=== Generating questions for each slot ===")
# for s in slots:
#     slot_id = s.get("slot_id")
#     target_marks = int(s.get("target_marks", 0))
#     print(f"\n{slot_id} | target_marks={target_marks}")

#     t = pick_template_for_mark(target_marks)
#     qtext, hits, fmt = generate_question(t, marks=target_marks, max_tries=6)

#     generated_questions.append({
#         "slot_id": slot_id,
#         "position": int(s.get("position", 0)),
#         "target_marks": target_marks,
#         "format": fmt,
#         "pattern_label": t.get("pattern_label"),
#         "template_source": {"pdf_stem": t.get("pdf_stem"), "question_id": t.get("question_id")},
#         "question_text": qtext,
#         "retrieved_sources": hits[:5],
#     })

# total_marks = sum(q["target_marks"] for q in generated_questions)
# expected_total = int(exam_blueprint.get("canonical_total_marks", 100))

# print("\n=== Summary ===")
# print("Generated questions:", len(generated_questions))
# print("Total marks:", total_marks, "| expected:", expected_total)

# # -------------------------------
# # 9) Save outputs
# # -------------------------------
# paper_json = {
#     "component": "generated_model_exam_paper_rag_v3",
#     "generator_model": FLAN_MODEL_NAME,
#     "device": device,
#     "retriever": {
#         "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
#         "faiss_index": str(SLIDES_FAISS_PATH),
#         "top_k_context": 15,
#         "context_max_chars": 2000
#     },
#     "exam_blueprint_source": str(BLUEPRINT_PATH),
#     "template_source": str(TEMPLATES_PATH),
#     "canonical_total_marks_expected": expected_total,
#     "canonical_total_marks_generated": total_marks,
#     "questions": sorted(generated_questions, key=lambda x: x["position"])
# }

# OUT_JSON.write_text(json.dumps(paper_json, indent=2, ensure_ascii=False), encoding="utf-8")

# lines = []
# lines.append("MODEL EXAM PAPER (RAG-generated v3)")
# lines.append(f"Generator: {FLAN_MODEL_NAME} | Device: {device}")
# lines.append(f"Total Marks: {total_marks} / {expected_total}")
# lines.append("=" * 70)

# for q in sorted(generated_questions, key=lambda x: x["position"]):
#     lines.append(f"\n{q['slot_id']} ({q['target_marks']} marks) — {q.get('pattern_label','')} — {q.get('format','')}")
#     lines.append(q["question_text"].strip())

# OUT_TXT.write_text("\n".join(lines).strip(), encoding="utf-8")

# print("\n✅ Saved:")
# print(" -", OUT_JSON)
# print(" -", OUT_TXT)

# print("\n--- Preview ---")
# print("\n".join(lines[:40]))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All required files found.
✅ Loaded FAISS index. vectors: 163
Device: cpu
Loading generator: google/flan-t5-base
✅ Generator loaded.

=== Generating questions for each slot ===

Q1 | target_marks=20

Q2 | target_marks=17

Q3 | target_marks=23

Q4 | target_marks=40

=== Summary ===
Generated questions: 4
Total marks: 100 | expected: 100

✅ Saved:
 - /content/drive/MyDrive/RP/model_exam_paper.json
 - /content/drive/MyDrive/RP/model_exam_paper.txt

--- Preview ---
MODEL EXAM PAPER (RAG-generated v3)
Generator: google/flan-t5-base | Device: cpu
Total Marks: 100 / 100

Q1 (20 marks) — ER_EER_MODELING — FALLBACK_GENERIC
Consider a realistic database scenario relevant to the given topic.
(a) Define the relations/entities and state primary keys and any constraints.
(b) Write the required SQL/ER/normalization steps to solve the problem.
(c) Briefly justify your desig

In [ ]:
# # ==========================================================
# # NOTEBOOK 5 — RAG MODEL PAPER GENERATION (OPENAI GPT)
# #
# # - Retrieval: SBERT + FAISS (from Notebook 4)
# # - Templates/Blueprint: from Notebook 2
# # - Generation: OpenAI Responses API (GPT)
# #
# # Output:
# #   /content/drive/MyDrive/RP/model_exam_paper.json
# #   /content/drive/MyDrive/RP/model_exam_paper.txt
# # ==========================================================

# # -------------------------------
# # 0) Mount Drive
# # -------------------------------
# from google.colab import drive
# drive.mount("/content/drive")

# # -------------------------------
# # 1) Install deps
# # -------------------------------
# !pip -q install sentence-transformers faiss-cpu openai

# import os, json, re, random, time
# from pathlib import Path
# import numpy as np
# import faiss
# from sentence_transformers import SentenceTransformer
# from openai import OpenAI

# # -------------------------------
# # 2) Paths (match Notebook 2/4 outputs)
# # -------------------------------
# RP_ROOT   = Path("/content/drive/MyDrive/RP")

# TEMPLATES_PATH = RP_ROOT / "template_questions.json"
# BLUEPRINT_PATH = RP_ROOT / "exam_blueprint_template.json"

# SLIDES_CHUNKS_PATH = RP_ROOT / "lecture_slides_extraction" / "slides_chunks.jsonl"
# SLIDES_META_PATH   = RP_ROOT / "slides_embeddings" / "slides_metadata.jsonl"
# SLIDES_FAISS_PATH  = RP_ROOT / "slides_embeddings" / "slides_faiss_index_flatip.index"

# OUT_JSON = RP_ROOT / "model_exam_paper.json"
# OUT_TXT  = RP_ROOT / "model_exam_paper.txt"

# for p in [TEMPLATES_PATH, BLUEPRINT_PATH, SLIDES_CHUNKS_PATH, SLIDES_META_PATH, SLIDES_FAISS_PATH]:
#     if not p.exists():
#         raise FileNotFoundError(f"Missing required file: {p}")

# print("✅ All required files found.")

# # -------------------------------
# # 3) OpenAI key + client
# # -------------------------------
# if not os.environ.get("OPENAI_API_KEY"):
#     os.environ["OPENAI_API_KEY"] = input("Paste OPENAI_API_KEY (will not be saved): ").strip()

# client = OpenAI()

# # Choose a generator model available to your account/project
# GEN_MODEL = "gpt-5.2"  # change if needed

# # Rate-limit safety
# MAX_RETRIES = 6
# BASE_BACKOFF = 2.0

# # -------------------------------
# # 4) Load artifacts
# # -------------------------------
# templates = json.loads(TEMPLATES_PATH.read_text(encoding="utf-8"))
# exam_blueprint = json.loads(BLUEPRINT_PATH.read_text(encoding="utf-8"))

# slide_chunks = [json.loads(l) for l in SLIDES_CHUNKS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
# slide_meta   = [json.loads(l) for l in SLIDES_META_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]

# index = faiss.read_index(str(SLIDES_FAISS_PATH))
# print("✅ Loaded FAISS index. vectors:", index.ntotal)

# # -------------------------------
# # 5) Retrieval embedding model (must match Notebook 4)
# # -------------------------------
# retriever_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# # -------------------------------
# # 6) Helpers
# # -------------------------------
# def normalize_l2(x: np.ndarray) -> np.ndarray:
#     x = x.astype("float32")
#     norm = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
#     return x / norm

# BAD_CONTEXT_PATTERNS = [
#     r"\[SOURCE:.*?\]",
#     r"---\s*SLIDE\s*\d+\s*---",
#     r"---\s*PAGE\s*\d+\s*---",
# ]

# def clean_context(text: str) -> str:
#     t = text or ""
#     for pat in BAD_CONTEXT_PATTERNS:
#         t = re.sub(pat, " ", t, flags=re.I | re.S)
#     t = re.sub(r'["\']{2,}', " ", t)
#     t = re.sub(r"[-_]{3,}", " ", t)
#     t = re.sub(r"\s+", " ", t).strip()
#     if len(t) > 2500:
#         t = t[:2500].rsplit(" ", 1)[0] + "..."
#     return t.strip()

# def retrieve_slide_context(query_text: str, top_k: int = 15, max_chars: int = 2000):
#     q_emb = retriever_model.encode([query_text]).astype("float32")
#     q_emb = normalize_l2(q_emb)
#     D, I = index.search(q_emb, top_k)

#     hits = []
#     parts = []
#     used_chars = 0

#     for idx, score in zip(I[0].tolist(), D[0].tolist()):
#         if idx < 0:
#             continue
#         chunk = slide_chunks[idx] if idx < len(slide_chunks) else {}
#         meta  = slide_meta[idx] if idx < len(slide_meta) else {}

#         raw = (chunk.get("text") or "").strip()
#         cleaned = clean_context(raw)

#         if not cleaned or len(cleaned.split()) < 18:
#             continue

#         hits.append({
#             "score": float(score),
#             "chunk_id": meta.get("chunk_id", chunk.get("chunk_id")),
#             "pdf_stem": meta.get("pdf_stem", chunk.get("pdf_stem")),
#             "slide_no": meta.get("slide_no", chunk.get("slide_no")),
#         })

#         parts.append(cleaned)
#         used_chars += len(cleaned) + 1
#         if used_chars >= max_chars:
#             break

#     context = "\n".join(parts).strip()
#     if len(context) > max_chars:
#         context = context[:max_chars].rsplit(" ", 1)[0] + "..."

#     return context, hits

# def looks_like_garbage(text: str) -> bool:
#     t = (text or "").strip()
#     if len(t) < 60:
#         return True
#     if "[SOURCE" in t or "--- SLIDE" in t or "--- PAGE" in t:
#         return True
#     alpha = sum(c.isalpha() for c in t)
#     if alpha / max(1, len(t)) < 0.35:
#         return True
#     if len(re.findall(r'["\']', t)) > 40:
#         return True
#     if len(re.findall(r"-", t)) > 80:
#         return True
#     return False

# def validate_structured(text: str) -> bool:
#     t = (text or "").strip()
#     has_parts = bool(re.search(r"\(a\)", t, flags=re.I) and re.search(r"\(b\)", t, flags=re.I))
#     return has_parts and (not looks_like_garbage(t))

# def enforce_clean_final(text: str) -> str:
#     t = (text or "").strip()
#     t = re.sub(r"\[SOURCE:.*?\]", " ", t, flags=re.I | re.S)
#     t = re.sub(r"---\s*SLIDE\s*\d+\s*---", " ", t, flags=re.I)
#     t = re.sub(r"---\s*PAGE\s*\d+\s*---", " ", t, flags=re.I)
#     t = re.sub(r"\s+", " ", t).strip()
#     return t

# def prompt_for_structured(template_text: str, context: str, pattern_label: str, marks: int):
#     return f"""
# You are a university lecturer creating ONE structured exam question for Database Management Systems worth about {marks} marks.

# STRICT RULES:
# - Use ONLY the CONTEXT facts. Do not invent additional theory not supported by context.
# - Do NOT copy sentences from TEMPLATE. Rewrite fully using new entity/table/attribute names and different numbers/values.
# - Output MUST include (a) and (b) subparts (and optionally (c)).
# - Output ONLY the question. No answers. No solutions.

# Pattern: {pattern_label}

# Required format:
# <Main question statement describing the scenario>
# (a) ...
# (b) ...
# (c) ... (optional)

# TEMPLATE (style reference only):
# {template_text}

# CONTEXT (must ground the question):
# {context}
# """.strip()

# def openai_generate_text(prompt: str, max_output_tokens: int = 450) -> str:
#     """
#     Uses OpenAI Responses API.
#     Docs: https://platform.openai.com/docs/api-reference/responses
#     """
#     last_err = None
#     for attempt in range(1, MAX_RETRIES + 1):
#         try:
#             resp = client.responses.create(
#                 model=GEN_MODEL,
#                 input=prompt,
#                 temperature=0.7,
#                 max_output_tokens=max_output_tokens,
#             )
#             # SDK helper exists in newer versions; fallback to parsing
#             if hasattr(resp, "output_text") and resp.output_text:
#                 return resp.output_text.strip()

#             out = ""
#             for item in getattr(resp, "output", []) or []:
#                 if getattr(item, "type", None) == "message":
#                     for c in getattr(item, "content", []) or []:
#                         if getattr(c, "type", None) == "output_text":
#                             out += getattr(c, "text", "") or ""
#             return (out or "").strip()

#         except Exception as e:
#             last_err = e
#             msg = str(e).lower()
#             if "rate limit" in msg or "429" in msg or "timeout" in msg or "temporarily" in msg or "server" in msg:
#                 backoff = min(60.0, BASE_BACKOFF * (2 ** (attempt - 1))) + random.uniform(0, 1.5)
#                 print(f"⚠️ OpenAI call failed (attempt {attempt}/{MAX_RETRIES}) → backoff {backoff:.1f}s | {e}")
#                 time.sleep(backoff)
#                 continue
#             raise
#     raise RuntimeError(f"OpenAI failed after retries: {last_err}")

# # -------------------------------
# # 7) Generate ONE question per slot (STRUCTURED ONLY)
# # -------------------------------
# def generate_question(template_obj: dict, marks: int, max_tries: int = 6):
#     template_text = (template_obj.get("full_text") or "").strip()
#     pattern_label = template_obj.get("pattern_label", "GENERAL_THEORY")

#     context, hits = retrieve_slide_context(template_text, top_k=15, max_chars=2000)
#     if not context:
#         context = clean_context(template_text)[:1200]

#     for attempt in range(1, max_tries + 1):
#         prompt = prompt_for_structured(template_text, context, pattern_label, marks)
#         gen = openai_generate_text(prompt, max_output_tokens=520)
#         gen = enforce_clean_final(gen)

#         if validate_structured(gen):
#             return gen, hits, "STRUCTURED"

#     safe = (
#         "Consider a realistic database scenario relevant to the given topic.\n"
#         "(a) Define the relations/entities and state primary keys and any constraints.\n"
#         "(b) Write the required SQL/ER/normalization steps to solve the problem.\n"
#         "(c) Briefly justify your design/queries."
#     )
#     return safe, hits, "FALLBACK_GENERIC"

# # -------------------------------
# # 8) Assemble full paper using blueprint slots
# # -------------------------------
# slots = exam_blueprint.get("question_slots", [])
# if not slots:
#     raise ValueError("exam_blueprint_template.json has no question_slots")

# templates_by_marks = {}
# for t in templates:
#     m = t.get("marks", None)
#     if m is None:
#         continue
#     templates_by_marks.setdefault(int(m), []).append(t)

# def pick_template_for_mark(mark: int):
#     pool = templates_by_marks.get(int(mark), [])
#     if pool:
#         return random.choice(pool)
#     return random.choice(templates)

# random.seed(42)

# generated_questions = []
# print("\n=== Generating questions for each slot ===")
# for s in slots:
#     slot_id = s.get("slot_id")
#     target_marks = int(s.get("target_marks", 0))
#     print(f"\n{slot_id} | target_marks={target_marks}")

#     t = pick_template_for_mark(target_marks)
#     qtext, hits, fmt = generate_question(t, marks=target_marks, max_tries=6)

#     generated_questions.append({
#         "slot_id": slot_id,
#         "position": int(s.get("position", 0)),
#         "target_marks": target_marks,
#         "format": fmt,
#         "pattern_label": t.get("pattern_label"),
#         "template_source": {"pdf_stem": t.get("pdf_stem"), "question_id": t.get("question_id")},
#         "question_text": qtext,
#         "retrieved_sources": hits[:5],
#     })

# total_marks = sum(q["target_marks"] for q in generated_questions)
# expected_total = int(exam_blueprint.get("canonical_total_marks", 100))

# print("\n=== Summary ===")
# print("Generated questions:", len(generated_questions))
# print("Total marks:", total_marks, "| expected:", expected_total)

# # -------------------------------
# # 9) Save outputs
# # -------------------------------
# paper_json = {
#     "component": "generated_model_exam_paper_rag_openai",
#     "generator_model": GEN_MODEL,
#     "retriever": {
#         "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
#         "faiss_index": str(SLIDES_FAISS_PATH),
#         "top_k_context": 15,
#         "context_max_chars": 2000
#     },
#     "exam_blueprint_source": str(BLUEPRINT_PATH),
#     "template_source": str(TEMPLATES_PATH),
#     "canonical_total_marks_expected": expected_total,
#     "canonical_total_marks_generated": total_marks,
#     "questions": sorted(generated_questions, key=lambda x: x["position"])
# }

# OUT_JSON.write_text(json.dumps(paper_json, indent=2, ensure_ascii=False), encoding="utf-8")

# lines = []
# lines.append("MODEL EXAM PAPER (RAG-generated — OpenAI)")
# lines.append(f"Generator: {GEN_MODEL}")
# lines.append(f"Total Marks: {total_marks} / {expected_total}")
# lines.append("=" * 70)

# for q in sorted(generated_questions, key=lambda x: x["position"]):
#     lines.append(f"\n{q['slot_id']} ({q['target_marks']} marks) — {q.get('pattern_label','')} — {q.get('format','')}")
#     lines.append(q["question_text"].strip())

# OUT_TXT.write_text("\n".join(lines).strip(), encoding="utf-8")

# print("\n✅ Saved:")
# print(" -", OUT_JSON)
# print(" -", OUT_TXT)

# print("\n--- Preview ---")
# print("\n".join(lines[:40]))


In [ ]:
!pip -q install --upgrade openai

In [ ]:
# ==========================================================
# NOTEBOOK 5 — AGENTIC RAG MODEL PAPER GENERATION (OPENAI GPT)
#
# Agent loop per slot:
#   PLAN (make retrieval query)
#   RETRIEVE (slides FAISS)
#   GENERATE (draft question)
#   JUDGE (score constraints)
#   REPAIR (re-retrieve + rewrite if needed)
#
# Output:
#   /content/drive/MyDrive/RP/model_exam_paper.json
#   /content/drive/MyDrive/RP/model_exam_paper.txt
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

!pip -q install sentence-transformers faiss-cpu openai

import os, json, re, random, time, math
from pathlib import Path
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# -------------------------------
# Paths
# -------------------------------
RP_ROOT   = Path("/content/drive/MyDrive/RP")

TEMPLATES_PATH = RP_ROOT / "template_questions.json"
BLUEPRINT_PATH = RP_ROOT / "exam_blueprint_template.json"

SLIDES_CHUNKS_PATH = RP_ROOT / "lecture_slides_extraction" / "slides_chunks.jsonl"
SLIDES_META_PATH   = RP_ROOT / "slides_embeddings" / "slides_metadata.jsonl"
SLIDES_FAISS_PATH  = RP_ROOT / "slides_embeddings" / "slides_faiss_index_flatip.index"

OUT_JSON = RP_ROOT / "model_exam_paper.json"
OUT_TXT  = RP_ROOT / "model_exam_paper.txt"

for p in [TEMPLATES_PATH, BLUEPRINT_PATH, SLIDES_CHUNKS_PATH, SLIDES_META_PATH, SLIDES_FAISS_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

print("✅ All required files found.")

# -------------------------------
# OpenAI
# -------------------------------
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = input("Paste OPENAI_API_KEY (will not be saved): ").strip()

client = OpenAI()
GEN_MODEL = "gpt-4.1"   # change if needed

MAX_RETRIES = 6
BASE_BACKOFF = 2.0

def openai_text(prompt: str, max_output_tokens: int = 600, temperature: float = 0.6) -> str:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.responses.create(
                model=GEN_MODEL,
                input=prompt,
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            if hasattr(resp, "output_text") and resp.output_text:
                return resp.output_text.strip()

            out = ""
            for item in getattr(resp, "output", []) or []:
                if getattr(item, "type", None) == "message":
                    for c in getattr(item, "content", []) or []:
                        if getattr(c, "type", None) == "output_text":
                            out += getattr(c, "text", "") or ""
            return (out or "").strip()
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            if "rate limit" in msg or "429" in msg or "timeout" in msg or "server" in msg or "temporarily" in msg:
                backoff = min(60.0, BASE_BACKOFF * (2 ** (attempt - 1))) + random.uniform(0, 1.5)
                print(f"⚠️ OpenAI failed (attempt {attempt}/{MAX_RETRIES}) → backoff {backoff:.1f}s | {e}")
                time.sleep(backoff)
                continue
            raise
    raise RuntimeError(f"OpenAI failed after retries: {last_err}")

# -------------------------------
# Load artifacts
# -------------------------------
templates = json.loads(TEMPLATES_PATH.read_text(encoding="utf-8"))
exam_blueprint = json.loads(BLUEPRINT_PATH.read_text(encoding="utf-8"))

slide_chunks = [json.loads(l) for l in SLIDES_CHUNKS_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
slide_meta   = [json.loads(l) for l in SLIDES_META_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
index = faiss.read_index(str(SLIDES_FAISS_PATH))
print("✅ Loaded FAISS index. vectors:", index.ntotal)

retriever_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# -------------------------------
# Retrieval helpers
# -------------------------------
def normalize_l2(x: np.ndarray) -> np.ndarray:
    x = x.astype("float32")
    norm = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / norm

BAD_CONTEXT_PATTERNS = [
    r"\[SOURCE:.*?\]",
    r"---\s*SLIDE\s*\d+\s*---",
    r"---\s*PAGE\s*\d+\s*---",
]

def clean_context(text: str) -> str:
    t = text or ""
    for pat in BAD_CONTEXT_PATTERNS:
        t = re.sub(pat, " ", t, flags=re.I | re.S)
    t = re.sub(r"\s+", " ", t).strip()
    if len(t) > 2500:
        t = t[:2500].rsplit(" ", 1)[0] + "..."
    return t.strip()

def retrieve_slide_context(query_text: str, top_k: int = 15, max_chars: int = 2200):
    q_emb = retriever_model.encode([query_text]).astype("float32")
    q_emb = normalize_l2(q_emb)
    D, I = index.search(q_emb, top_k)

    hits, parts = [], []
    used = 0

    for idx, score in zip(I[0].tolist(), D[0].tolist()):
        if idx < 0:
            continue
        chunk = slide_chunks[idx] if idx < len(slide_chunks) else {}
        meta  = slide_meta[idx] if idx < len(slide_meta) else {}

        raw = (chunk.get("text") or "").strip()
        cleaned = clean_context(raw)

        if not cleaned or len(cleaned.split()) < 18:
            continue

        hits.append({
            "score": float(score),
            "chunk_id": meta.get("chunk_id", chunk.get("chunk_id")),
            "pdf_stem": meta.get("pdf_stem", chunk.get("pdf_stem")),
            "slide_no": meta.get("slide_no", chunk.get("slide_no")),
        })

        parts.append(cleaned)
        used += len(cleaned) + 1
        if used >= max_chars:
            break

    context = "\n".join(parts).strip()
    if len(context) > max_chars:
        context = context[:max_chars].rsplit(" ", 1)[0] + "..."
    return context, hits

# -------------------------------
# Anti-copy similarity (simple + effective)
# -------------------------------
def ngram_set(text: str, n: int = 3):
    toks = re.findall(r"[A-Za-z0-9_]+", (text or "").lower())
    return set(tuple(toks[i:i+n]) for i in range(0, max(0, len(toks)-n+1)))

def trigram_overlap(a: str, b: str) -> float:
    A = ngram_set(a, 3)
    B = ngram_set(b, 3)
    if not A or not B:
        return 0.0
    return len(A & B) / max(1, len(A))

# -------------------------------
# Output validation
# -------------------------------
def looks_like_garbage(text: str) -> bool:
    t = (text or "").strip()
    if len(t) < 80: return True
    if "[SOURCE" in t or "--- SLIDE" in t or "--- PAGE" in t: return True
    alpha = sum(c.isalpha() for c in t)
    if alpha / max(1, len(t)) < 0.35: return True
    return False

def has_structured_parts(text: str) -> bool:
    t = (text or "")
    return bool(re.search(r"\(a\)", t, flags=re.I) and re.search(r"\(b\)", t, flags=re.I))

def enforce_clean_final(text: str) -> str:
    t = (text or "").strip()
    for pat in BAD_CONTEXT_PATTERNS:
        t = re.sub(pat, " ", t, flags=re.I | re.S)
    t = re.sub(r"\s+", " ", t).strip()
    return t

# -------------------------------
# Agent steps: PLAN / GENERATE / JUDGE / REPAIR
# -------------------------------
def plan_retrieval_query(template_text: str, pattern_label: str) -> str:
    prompt = f"""
You will create a short retrieval query for lecture slides about DBMS.
Return ONLY one line query (no quotes), 6 to 14 words.

Pattern: {pattern_label}
Template:
{template_text}
""".strip()
    q = openai_text(prompt, max_output_tokens=60, temperature=0.2)
    q = re.sub(r"\s+", " ", q).strip()
    return q[:120]

def prompt_for_generation(template_text: str, context: str, pattern_label: str, marks: int) -> str:
    return f"""
You are a university lecturer creating ONE structured exam question for Database Management Systems worth about {marks} marks.

STRICT RULES:
- Use ONLY the CONTEXT facts. Do not add theory not supported by context.
- Do NOT copy sentences from TEMPLATE. Use new entity/table/attribute names and different numbers/values.
- Output MUST include (a) and (b) (optional (c)).
- Output ONLY the question. No answers.

Pattern: {pattern_label}

Format:
<Scenario statement>
(a) ...
(b) ...
(c) ... (optional)

TEMPLATE (style only):
{template_text}

CONTEXT:
{context}
""".strip()

def judge_question(template_text: str, context: str, generated: str, marks: int) -> dict:
    # GPT-as-judge returns JSON with score + reasons
    prompt = f"""
You are grading whether an exam question meets constraints. Return ONLY JSON.

Constraints:
1) Has (a) and (b).
2) Not garbage.
3) Not copying template (should be rewritten; different entities/values).
4) Grounded in context (should match context topic; no unrelated content).
5) Reasonable difficulty for ~{marks} marks.

Template:
{template_text}

Context:
{context}

Generated:
{generated}

Return JSON exactly like:
{{
  "pass": true/false,
  "score": 0-100,
  "reasons": ["...","..."],
  "needs": ["what to fix", "..."]
}}
""".strip()
    raw = openai_text(prompt, max_output_tokens=220, temperature=0.1)
    try:
        j = json.loads(raw)
        if not isinstance(j, dict):
            raise ValueError("judge not dict")
        return j
    except Exception:
        # fallback judge if JSON breaks
        return {
            "pass": False,
            "score": 0,
            "reasons": ["Judge JSON parse failed"],
            "needs": ["Regenerate with stricter instructions"]
        }

def repair_prompt(generated: str, needs: list, context: str, marks: int) -> str:
    needs_txt = "; ".join(needs[:6]) if needs else "Improve structure and grounding."
    return f"""
Rewrite the exam question to fix these issues: {needs_txt}

STRICT:
- Keep it a single question with (a) and (b) (optional (c)).
- Use ONLY the CONTEXT facts.
- Do NOT include answers.
- Keep difficulty ~{marks} marks.

CONTEXT:
{context}

CURRENT QUESTION:
{generated}

Return ONLY the revised question.
""".strip()

# -------------------------------
# Main agent loop per slot
# -------------------------------
def generate_agentic_question(template_obj: dict, marks: int, max_rounds: int = 4):
    template_text = (template_obj.get("full_text") or "").strip()
    pattern_label = template_obj.get("pattern_label", "GENERAL_THEORY")

    # PLAN
    query = plan_retrieval_query(template_text, pattern_label)

    # RETRIEVE initial
    context, hits = retrieve_slide_context(query, top_k=15, max_chars=2200)
    if not context:
        context, hits = retrieve_slide_context(template_text[:400], top_k=20, max_chars=2200)

    draft = None
    judge = None

    for rnd in range(1, max_rounds + 1):
        # GENERATE
        if draft is None:
            prompt = prompt_for_generation(template_text, context, pattern_label, marks)
            draft = openai_text(prompt, max_output_tokens=520, temperature=0.65)
        else:
            # REPAIR draft
            prompt = repair_prompt(draft, (judge or {}).get("needs", []), context, marks)
            draft = openai_text(prompt, max_output_tokens=520, temperature=0.55)

        draft = enforce_clean_final(draft)

        # Hard checks
        if looks_like_garbage(draft) or (not has_structured_parts(draft)):
            judge = {"pass": False, "score": 0, "reasons": ["Failed basic structure"], "needs": ["Add (a) and (b), fix formatting"]}
        else:
            # anti-copy
            overlap = trigram_overlap(template_text, draft)
            if overlap > 0.18:
                judge = {"pass": False, "score": 35, "reasons": [f"Too similar to template (overlap={overlap:.2f})"],
                         "needs": ["Rewrite with different scenario/entities/values; avoid template phrasing"]}
            else:
                # GPT judge
                judge = judge_question(template_text, context, draft, marks)

        if judge.get("pass") and judge.get("score", 0) >= 75:
            return draft, hits, "AGENTIC_OK", query, judge

        # If fail, improve retrieval once mid-way
        if rnd == 2:
            expanded_query = f"{query} {pattern_label} keys constraints SQL"
            context2, hits2 = retrieve_slide_context(expanded_query, top_k=22, max_chars=2400)
            if context2 and len(context2) > len(context):
                context, hits = context2, hits2

    # final fallback (safe generic)
    safe = (
        "Consider a realistic database scenario relevant to the given topic.\n"
        "(a) Define the relations/entities and state primary keys and any constraints.\n"
        "(b) Write the required SQL/ER/normalization steps to solve the problem.\n"
        "(c) Briefly justify your design/queries."
    )
    return safe, hits, "FALLBACK_GENERIC", query, judge

# -------------------------------
# Assemble paper
# -------------------------------
slots = exam_blueprint.get("question_slots", [])
if not slots:
    raise ValueError("exam_blueprint_template.json has no question_slots")

templates_by_marks = {}
for t in templates:
    m = t.get("marks", None)
    if m is None:
        continue
    templates_by_marks.setdefault(int(m), []).append(t)

def pick_template_for_mark(mark: int):
    pool = templates_by_marks.get(int(mark), [])
    return random.choice(pool) if pool else random.choice(templates)

random.seed(42)

generated_questions = []
print("\n=== Generating questions (AGENTIC) ===")
for s in slots:
    slot_id = s.get("slot_id")
    target_marks = int(s.get("target_marks", 0))
    print(f"\n{slot_id} | target_marks={target_marks}")

    t = pick_template_for_mark(target_marks)
    qtext, hits, fmt, query, judge = generate_agentic_question(t, marks=target_marks, max_rounds=4)

    generated_questions.append({
        "slot_id": slot_id,
        "position": int(s.get("position", 0)),
        "target_marks": target_marks,
        "format": fmt,
        "pattern_label": t.get("pattern_label"),
        "template_source": {"pdf_stem": t.get("pdf_stem"), "question_id": t.get("question_id")},
        "retrieval_query": query,
        "judge": judge,
        "question_text": qtext,
        "retrieved_sources": hits[:8],
    })

total_marks = sum(q["target_marks"] for q in generated_questions)
expected_total = int(exam_blueprint.get("canonical_total_marks", 100))

print("\n=== Summary ===")
print("Generated questions:", len(generated_questions))
print("Total marks:", total_marks, "| expected:", expected_total)

# -------------------------------
# Save outputs
# -------------------------------
paper_json = {
    "component": "generated_model_exam_paper_agentic_rag_openai",
    "generator_model": GEN_MODEL,
    "retriever": {
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
        "faiss_index": str(SLIDES_FAISS_PATH),
        "top_k_context": 15,
        "context_max_chars": 2200
    },
    "exam_blueprint_source": str(BLUEPRINT_PATH),
    "template_source": str(TEMPLATES_PATH),
    "canonical_total_marks_expected": expected_total,
    "canonical_total_marks_generated": total_marks,
    "questions": sorted(generated_questions, key=lambda x: x["position"])
}

OUT_JSON.write_text(json.dumps(paper_json, indent=2, ensure_ascii=False), encoding="utf-8")

lines = []
lines.append("MODEL EXAM PAPER (Agentic RAG — OpenAI)")
lines.append(f"Generator: {GEN_MODEL}")
lines.append(f"Total Marks: {total_marks} / {expected_total}")
lines.append("=" * 70)

for q in sorted(generated_questions, key=lambda x: x["position"]):
    lines.append(f"\n{q['slot_id']} ({q['target_marks']} marks) — {q.get('pattern_label','')} — {q.get('format','')}")
    lines.append(q["question_text"].strip())

OUT_TXT.write_text("\n".join(lines).strip(), encoding="utf-8")

print("\n✅ Saved:")
print(" -", OUT_JSON)
print(" -", OUT_TXT)

print("\n--- Preview ---")
print("\n".join(lines[:40]))
